In [1]:
import pandas as pd
import numpy as np
from matminer.datasets import load_dataset
from pymatgen.core.composition import Composition

class DataLoader:
    def __init__(self, dataset_name, n_samples):
        self.dataset_name = dataset_name
        self.n_samples = n_samples
        self.data = self.load_data()

    def load_data(self):
        data = load_dataset(self.dataset_name)['composition'].to_frame().iloc[:self.n_samples]
        data = data.drop_duplicates().reset_index(drop=True)
        return data

    def create_composition(self, formula):
        try:
            return Composition(formula)
        except ValueError:
            print(f"Error parsing formula: {formula}")
            return None

    def prepare_compositions(self):
        self.data["_Composition"] = self.data['composition'].apply(self.create_composition)
        self.data = self.data.dropna(subset=["_Composition"]).reset_index(drop=True)
        return self.data

In [2]:
from matminer.featurizers.composition import ElementProperty, ElementFraction

class FeatureExtractor:
    def __init__(self):
        self.ep_featurizer = ElementProperty.from_preset('magpie')
        self.ef_featurizer = ElementFraction()

    def featurize(self, data):
        data = data.dropna()
        ep_ftd = self.ep_featurizer.featurize_dataframe(data, col_id='_Composition', ignore_errors=True)
        ef_ftd = self.ef_featurizer.featurize_dataframe(data, col_id='_Composition', ignore_errors=True)
        return ep_ftd, ef_ftd

In [3]:
import pickle
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

class ModelManager:
    def __init__(self, model_paths):
        self.models = self.load_models(model_paths)

    def load_models(self, model_paths):
        models = {}
        for name, path in model_paths.items():
            with open(path, 'rb') as file:
                models[name] = pickle.load(file)
        return models

    def predict(self, model_name, X):
        return self.models[model_name].predict(X)

    def evaluate(self, y_true, y_pred):
        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred)
        recall = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        roc_auc = roc_auc_score(y_true, y_pred)
        conf_matrix = confusion_matrix(y_true, y_pred)

        return {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "roc_auc": roc_auc,
            "conf_matrix": conf_matrix
        }

In [4]:
class SuperconductorPredictor:
    def __init__(self, data_loader, feature_extractor, model_manager):
        self.data_loader = data_loader
        self.feature_extractor = feature_extractor
        self.model_manager = model_manager

    def preprocess_data(self, ep_ftd, ef_ftd):
        def preprocess_for_model1(ep_ftd):
            ep_ftd["having_tc"] = (ep_ftd["Critical Temp"] >= 10).astype(int)
            ep_X = ep_ftd.iloc[:, 4:-1]
            ep_y = ep_ftd['having_tc']
            return ep_X, ep_y

        def preprocess_for_model3(ef_ftd):
            ef_ftd["having_tc"] = (ef_ftd["Critical Temp"] >= 10).astype(int)
            ef_X = ef_ftd.iloc[:, 4:-1]
            ef_y = ef_ftd['having_tc']
            return ef_X, ef_y

        def preprocess_for_model2(ep_ftd, ef_ftd):
            ef_ftd = ef_ftd.iloc[:, 2:]
            ep_ftd = ep_ftd.iloc[:, 2:]
            
            merged_df = pd.merge(ef_ftd, ep_ftd, left_on=["Critical Temp", "_Composition"], right_on=["Critical Temp", "_Composition"], how="inner")
            merged_df["having_tc"] = (merged_df["Critical Temp"] >= 10).astype(int)

            efep_X = merged_df.iloc[:, 2:-1]
            efep_y = merged_df['having_tc']    
            return efep_X, efep_y, merged_df

        ep_X, ep_y = preprocess_for_model1(ep_ftd)
        efep_X, efep_y, _ = preprocess_for_model2(ep_ftd, ef_ftd)
        ef_X, ef_y = preprocess_for_model3(ef_ftd)
        
        return ep_X, ep_y, efep_X, efep_y, ef_X, ef_y

    def ensemble_predict(self, ep_ftd, ef_ftd):
        ep_X, ep_y, efep_X, efep_y, ef_X, ef_y = self.preprocess_data(ep_ftd, ef_ftd)
        
        pred_ep = self.model_manager.predict('model_ep', ep_X)
        pred_efep = self.model_manager.predict('model_efep', efep_X)
        pred_ef = self.model_manager.predict('model_ef', ef_X)
        
        # Combine predictions using majority vote
        n_samples = len(pred_ep)
        ensemble_pred = np.zeros(n_samples)
        for i in range(n_samples):
            class_counts = np.bincount([pred_ep[i], pred_efep[i], pred_ef[i]])
            ensemble_pred[i] = np.argmax(class_counts)

        ensemble_pred = ensemble_pred.astype(int)
        return ensemble_pred, ep_y, efep_y, ef_y, ef_ftd

    def evaluate_ensemble(self, ensemble_pred, ep_y, efep_y, ef_y):
        eval_results = self.model_manager.evaluate(efep_y, ensemble_pred)
        return eval_results

    def create_result_dataframe(self, ensemble_pred, ef_ftd):
        result_df = pd.DataFrame({
            "composition": ef_ftd["_Composition"],
            "Tc": ef_ftd["Critical Temp"],
            "prediction": ensemble_pred
        })
        return result_df

In [ ]:
if __name__ == "__main__":
    # Paths to the pre-trained models
    model_paths = {
        'model_ep': 'sc_ep_rf_cl.pkl',
        'model_efep': 'sc_efep_rf_cl.pkl',
        'model_ef': 'sc_ef_rf_cl.pkl'
    }

    # Initialize the components
    data_loader = DataLoader("superconductivity2018", 10000)
    feature_extractor = FeatureExtractor()
    model_manager = ModelManager(model_paths)
    predictor = SuperconductorPredictor(data_loader, feature_extractor, model_manager)

    # Load and preprocess data
    data = data_loader.prepare_compositions()

    # Featurize the data
    ep_ftd, ef_ftd = feature_extractor.featurize(data)

    # Ensemble prediction
    ensemble_pred, ep_y, efep_y, ef_y, ef_ftd = predictor.ensemble_predict(ep_ftd, ef_ftd)

    # Evaluate the ensemble model
    results = predictor.evaluate_ensemble(ensemble_pred, ep_y, efep_y, ef_y)

    # Print results
    print("Evaluation Results:")
    for metric, value in results.items():
        print(f"{metric}: {value}")

    # Create and print the result DataFrame
    result_df = predictor.create_result_dataframe(ensemble_pred, ef_ftd)
    print(result_df.head())

C:\Users\hp\anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.2 when using version 1.5.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\hp\anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.2 when using version 1.5.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Error parsing formula: Eu1.45Pr0.05Ce0.5Sr2Cu2Nb1O10=z
Error parsing formula: Sm1Ba-1Cu3O6.94
Error parsing formula: Y2C2Br0.5!1.5
Error parsing formula: Hg0.3Pb0.7Sr1.75La0.25Cu1O4+2


ElementProperty:   0%|          | 0/9996 [00:00<?, ?it/s]